# Commodities Vol & Relative-Value Dashboard

A focused signal monitor for commodity **volatility** and **relative value**. It scans a
liquid commodity universe and surfaces five kinds of trade candidate, each with an explicit
action and a plain-English reason.

**How to use it**
1. Run every cell top to bottom.
2. Click **Load / refresh data** (pulls prices + implied vol from Bloomberg).
3. The controls apply instantly — no separate "recompute" step.

> These are systematic *candidates*, not advice. Always check liquidity, curve, event risk,
> option depth and desk limits before trading.


## What each number means

**Implied volatility (IV)** — the option market's *forward-looking* estimate of how volatile the
asset will be over roughly the option's life (here ~30‑day ATM, Bloomberg `hist_put_imp_vol`).
It is **not** today's price move and **not** past volatility; it is what the market is *betting*
future volatility will be. IV is quoted in annualized vol points (%).

**Realized volatility (RV)** — *backward-looking* volatility actually delivered by the price,
annualized (×√252) to vol points so it is directly comparable to IV. There are several ways to
estimate it, and the choice changes how fast it reacts — pick one in the **Realized-vol estimator**
control:

| Estimator | Uses | Character |
|---|---|---|
| Close-to-close | closes only | Standard; a single big day is diluted across the window |
| EWMA (RiskMetrics, λ=0.94) | closes | Weights recent days heavily → **reacts fast** to a shock |
| Parkinson | high/low | Uses intraday range; more efficient, ignores gaps |
| Garman-Klass | OHLC | Range + open/close; efficient |
| Yang-Zhang | OHLC | Range + overnight gaps; most complete |

The **window** — **1 week** (5 trading days), **1 month** (21) or **3 months** (63) — sets how much
history feeds the estimate. *If a huge one-day move barely moves RV, use 1 week or EWMA* — a
3-month window buries a single day.

**IV − RV (variance risk premium)** — how much more (or less) the option market is charging vs what
the asset is actually delivering. Persistently positive is the classic "sell vol" premium.

**Vol data quality** — computed, not hand-assigned. It is the fraction of recent days the IV series
*actually moves*: **Good** ≥ 50%, **Fair** 35–50%, **Weak** < 35%. These cut-offs are calibrated to
this feed — on Bloomberg's commodity IV even very liquid names (WTI, Brent, gold) only re-print a
fresh mark on ~50–70% of days, so a stricter bar made almost everything read Fair. Genuinely
illiquid options (many softs, palladium, steel) update rarely, so their IV looks flat for days —
that is why they still read Weak/stale and are filtered out by default.

## Confidence — what the % actually is

Every signal carries a **confidence %** and a sample size **n**. It is not a hand-tuned score. For
each signal the notebook asks one question of that asset's own 3-year history:

> *On past days when this exact setup was **at least as extreme** as it is today, how often did the
> thing we're betting on actually happen over the next N days?*

The raw hit rate is then shrunk and adjusted:

```
raw   = wins / n                     empirical hit rate of comparable past setups
p     = (wins + 10) / (n + 20)       Beta(10,10) posterior mean  ->  shrinks small n toward 50%
conf  = 50 + (p*100 - 50) * q        q = 1.00 Good, 0.85 Fair, 0.65 Weak IV data
conf  = clip(conf, 20, 92)           never claim certainty in either direction
```

The shrinkage is what stops a 4-for-4 fluke reading as 100%: with n=4 the number barely moves off
50. The quality factor `q` pulls the number back toward a coin flip when the IV series feeding the
signal is stale or thin. Because the historical episodes are filtered to be *at least as extreme as
today*, a more stretched signal is automatically scored against a more selective (and usually
smaller) comparison set — extremity and sample size trade off against each other on their own.

**What each engine counts as a "win"** (the horizon N in brackets):

| Engine | Win condition |
|---|---|
| IV mean-reversion [21d] | IV higher 21d later (if bought cheap) / lower (if sold rich) |
| Variance risk premium [= RV window] | Realized vol over the *following* window came in below the implied quoted that day (sell) / above it (buy) |
| Vol dispersion [21d] | The IV spread moved back toward its own mean |
| Correlation RV [20d] | The performance gap between the two legs closed |
| Lead-lag catch-up [= the lead, in days] | The follower moved the same way the leader had |

**Reading it:** 50% = coin flip, no historical edge. **>60%** is a genuine edge worth sizing.
**<45%** means this setup historically resolved the *other* way — treat it as a warning, not a
trade. Always look at **n**: a 70% with n=8 is far weaker evidence than a 58% with n=120. The
percentile-rank and z-score inputs behind the historical episodes are all computed on *trailing*
windows, so the hit rates carry no look-ahead. They are still in-sample statistics on three years
of one asset — a sanity check on the setup, not a backtested P&L.

## The five engines

1. **IV mean-reversion** — IV cheap/rich vs *its own* history (percentile). Buy vol when cheap,
   sell when rich.
2. **Variance risk premium** — IV vs realized. Sell vol when IV is richer than delivered vol; buy
   when it is not. (The live signal compares today's IV to *recent* realized; only the confidence
   backtest looks forward, and only over history that has already happened.)
3. **Vol dispersion (pairs)** — the *spread of two related assets' vols* breaking out of its normal
   band. E.g. WTI vol vs Brent vol. Sell the relatively rich vol, buy the cheap one, expect the
   spread to converge.
4. **Correlation RV** — a **convergence (relative-value)** trade on two normally-correlated
   assets. When their 60-day correlation is stretched from its own norm (either a *breakdown* —
   they've decoupled — or a *spike* — an unusually tight common-factor regime), we fade the recent
   **relative-performance gap**: **BUY the underperformer, SELL the outperformer**, betting the pair
   re-converges. *This is mean-reversion on the spread — it is NOT a momentum bet, and it is
   deliberately different from engine 5.* Worked example: gold −1.9% vs platinum −1.1% — gold has
   fallen more, so gold is the cheap/underperforming leg → **buy gold, sell platinum**, expecting
   the gap to close.
5. **Lead-lag catch-up** — a **directional follow-through** trade, the opposite premise to engine 4.
   One asset reliably *leads* another by a few days; the leader has already moved and the laggard
   hasn't yet, so you trade the laggard in the **same direction** the leader went (if the leader
   rallied, buy the laggard) expecting it to catch up. Convergence sells the leg that ran; catch-up
   follows it. The **Lead-lag chart** on the Correlation & Pairs tab shows this directly: the
   leader re-drawn shifted forward by its measured lead should overlay the follower, and the
   cross-correlation bars show which lag actually wins (a flat bar profile = no timing edge).

**Pair scope.** Correlation and lead-lag run **within an asset class** (energy vs energy, metals vs
metals, ags vs ags) **plus anything against the macro overlays** — S&P 500, the dollar (DXY) and
the US 10y note future. Cross-*commodity*-class pairs (silver vs soybean meal) stay excluded; those
were the noise. Macro pairs need only |avg corr| ≥ 0.30 to qualify vs 0.45 within a class, because
cross-asset links are real but structurally looser.

> **On the 10y:** the ticker is the 10-year **note future (TY1)**, not the yield. Return correlation
> needs a price series — the % change of a yield is unstable — so read "US 10Y" going **up** as
> yields going **down**. SPX, DXY and TY1 are price-only: no implied vol is pulled for them, so they
> appear in correlation and lead-lag signals but never in the vol engines.

## Buying and selling vol

- **Buy vol** (expect vol to rise): long straddle/strangle, long call or put, calendar, debit
  spread, or a variance swap. Long straddle = long call + long put same strike/expiry → **long
  gamma**, profits from big moves in either direction.
- **Sell vol** (expect vol to fall): short straddle/strangle, iron fly/condor, credit spread, or
  short variance. **Short premium** = collect option premium; spreads cap the downside vs naked
  shorts.


In [ ]:
# =====================================================================
# CONFIG — universe, asset classes, theme, fixed parameters
# =====================================================================
import bql
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets

bq = bql.Service()

# Commodity universe grouped by asset class, plus a small Macro block.
# Macro names are price-only (no implied vol pulled) and exist so energy /
# metals can be correlated and lead-lag tested against rates, the dollar and
# equities — the cross-asset drivers that actually move commodities.
UNIVERSE = {
    "Energy": {
        "WTI Crude": "CL1 Comdty", "Brent Crude": "CO1 Comdty",
        "Natural Gas": "NG1 Comdty", "RBOB Gasoline": "XB1 Comdty",
        "Heating Oil": "HO1 Comdty",
    },
    "Precious Metals": {
        "Gold": "GC1 Comdty", "Silver": "SI1 Comdty",
        "Platinum": "PL1 Comdty", "Palladium": "PA1 Comdty",
    },
    "Base Metals": {
        "Copper": "HG1 Comdty", "Steel HRC": "HRC1 Comdty",
        "Aluminium LME": "LMAHDS03 Comdty", "Nickel LME": "LMNIDS03 Comdty",
        "Iron Ore SGX": "TIO1 Comdty",
    },
    "Grains / Oilseeds": {
        "Corn": "C 1 Comdty", "Wheat CBOT": "W 1 Comdty",
        "Wheat Kansas": "KW1 Comdty", "Soybeans": "S 1 Comdty",
        "Soybean Oil": "BO1 Comdty", "Soybean Meal": "SM1 Comdty",
    },
    "Softs": {
        "Sugar": "SB1 Comdty", "Coffee": "KC1 Comdty",
        "Cotton": "CT1 Comdty", "Cocoa": "CC1 Comdty",
    },
    "Livestock": {
        "Live Cattle": "LC1 Comdty", "Lean Hogs": "LH1 Comdty",
    },
    # Price-only cross-asset overlays. The 10y here is the NOTE FUTURE, not the
    # yield: return correlation needs a price series (a yield's % change is
    # unstable near zero), so read "US 10Y" rising as yields FALLING.
    "Macro": {
        "S&P 500": "SPX Index", "US Dollar (DXY)": "DXY Curncy",
        "US 10Y (note fut)": "TY1 Comdty",
    },
}

MACRO_CLASS = "Macro"

# Assets with reliable listed option (implied-vol) markets. Price-only names
# (LME/SGX and the whole macro block) still participate in correlation/lead-lag.
NO_IV = {"LMAHDS03 Comdty", "LMNIDS03 Comdty", "TIO1 Comdty",
         "SPX Index", "DXY Curncy", "TY1 Comdty"}

CLASS_ORDER = ["Energy", "Precious Metals", "Base Metals",
               "Grains / Oilseeds", "Softs", "Livestock", "Macro"]

NAME, ASSET_CLASS = {}, {}
for cls, mp in UNIVERSE.items():
    for n, t in mp.items():
        NAME[t] = n
        ASSET_CLASS[t] = cls

ALL_TICKERS = list(NAME)
IV_TICKERS = [t for t in ALL_TICKERS if t not in NO_IV]

# Fixed analytics windows (the tunable ones live in the control panel).
HISTORY_YEARS   = 3
CORR_WINDOW     = 60     # rolling correlation window (days)
PAIR_WIN        = 20     # recent-return window for pair convergence trades
LEADLAG_WINDOW  = 90     # window for lead-lag estimation
LEADLAG_MAX     = 10     # max lead/lag tested, in days
CONF_HORIZON    = 21     # forward horizon (days) for scoring vol-signal confidence

# Minimum |average correlation| for a pair to count as "normally related".
# Macro-vs-commodity links are real but structurally looser than
# commodity-vs-commodity, so they get a lower bar or nothing would ever print.
MIN_MEAN_CORR       = 0.45
MIN_MEAN_CORR_MACRO = 0.30

# High-contrast dark theme.
BG, PANEL, GRID = "#0B0E14", "#151B26", "#2C3644"
TXT, MUTED = "#F2F6FC", "#95A3B8"
GREEN, RED, AMBER = "#25D07A", "#FF5B5B", "#FFC44D"
BLUE, PURPLE, TEAL = "#4DB6FF", "#C58CFF", "#2FD9C6"

ENGINE_ORDER = [
    ("IV mean-reversion",      BLUE),
    ("Variance risk premium",  AMBER),
    ("Vol dispersion (pairs)", TEAL),
    ("Correlation RV",         PURPLE),
    ("Lead-lag catch-up",      GREEN),
]

STATE = {}

In [ ]:
# =====================================================================
# ANALYTICS — volatility estimators, percentile/band stats, data
# quality, lead-lag, confidence scoring, and the five signal engines.
# Pure pandas/numpy; uses NAME / ASSET_CLASS defined above.
# =====================================================================
ANN = 252.0


# =============================================================================
# Volatility estimators
# =============================================================================
def _log(df):
    return np.log(df.astype(float))

def rv_close_to_close(px, window):
    r = _log(px["close"]).diff()
    return r.rolling(window).std() * np.sqrt(ANN) * 100.0

def rv_ewma(px, window, lam=0.94):
    r = _log(px["close"]).diff()
    var = (r ** 2).ewm(alpha=1 - lam, adjust=False).mean()
    return np.sqrt(var * ANN) * 100.0

def rv_parkinson(px, window):
    hl = _log(px["high"]) - _log(px["low"])
    var = (1.0 / (4.0 * np.log(2.0))) * (hl ** 2)
    return np.sqrt(var.rolling(window).mean() * ANN) * 100.0

def rv_garman_klass(px, window):
    hl = _log(px["high"]) - _log(px["low"])
    co = _log(px["close"]) - _log(px["open"])
    term = 0.5 * hl ** 2 - (2 * np.log(2) - 1) * co ** 2
    return np.sqrt(term.rolling(window).mean().clip(lower=0) * ANN) * 100.0

def _rogers_satchell_term(px):
    ho = _log(px["high"]) - _log(px["open"])
    hc = _log(px["high"]) - _log(px["close"])
    lo = _log(px["low"]) - _log(px["open"])
    lc = _log(px["low"]) - _log(px["close"])
    return ho * hc + lo * lc

def rv_yang_zhang(px, window):
    o, c = _log(px["open"]), _log(px["close"])
    prev_c = c.shift(1)
    overnight = o - prev_c               # close -> open
    openclose = c - o                    # open -> close
    var_o = overnight.rolling(window).var()
    var_c = openclose.rolling(window).var()
    var_rs = _rogers_satchell_term(px).rolling(window).mean()
    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    var = var_o + k * var_c + (1 - k) * var_rs
    return np.sqrt(var.clip(lower=0) * ANN) * 100.0

RV_ESTIMATORS = {
    "Close-to-close": rv_close_to_close,
    "EWMA (RiskMetrics)": rv_ewma,
    "Parkinson (H-L)": rv_parkinson,
    "Garman-Klass (OHLC)": rv_garman_klass,
    "Yang-Zhang (OHLC)": rv_yang_zhang,
}

def realized_vol(px, estimator, window):
    fn = RV_ESTIMATORS.get(estimator, rv_close_to_close)
    try:
        return fn(px, window)
    except Exception:
        return rv_close_to_close(px, window)

# =============================================================================
# Percentile / z helpers
# =============================================================================
def pct_rank(series, lookback):
    s = series.dropna()
    if len(s) < 30:
        return np.nan, np.nan, np.nan
    s = s.iloc[-lookback:] if lookback else s
    v = s.iloc[-1]
    pr = (s < v).mean() * 100.0
    sd = s.std()
    z = (v - s.mean()) / sd if sd and pd.notna(sd) else np.nan
    return v, pr, z

def band_z(series, lookback):
    """Current value vs its own rolling mean/std band -> (cur, mean, sd, z)."""
    s = series.dropna()
    if len(s) < 40:
        return np.nan, np.nan, np.nan, np.nan
    s = s.iloc[-lookback:] if lookback else s
    cur, mean, sd = s.iloc[-1], s.mean(), s.std()
    z = (cur - mean) / sd if sd and pd.notna(sd) else np.nan
    return cur, mean, sd, z

def recent_return(close, tk, window):
    if tk not in close:
        return np.nan
    s = close[tk].dropna()
    if len(s) < window + 1:
        return np.nan
    return s.pct_change(window).iloc[-1] * 100.0

def _win(lookback, default=756):
    """Turn the lookback control (252/504/0=all) into a usable rolling window."""
    return int(lookback) if lookback else default

def rolling_pct_rank(s, lookback):
    """Percentile rank of each point within its own trailing window (0-100).

    Used only for building historical 'comparable setups' — every point is ranked
    against data available AT THAT TIME, so the confidence backtest has no
    look-ahead. (The displayed percentile still comes from pct_rank above.)
    """
    w = max(60, min(_win(lookback), max(len(s), 60)))
    mp = max(30, w // 4)
    try:
        return s.rolling(w, min_periods=mp).rank(pct=True) * 100.0
    except Exception:      # pandas < 1.4 has no Rolling.rank
        return s.rolling(w, min_periods=mp).apply(
            lambda x: (x[:-1] < x[-1]).mean() * 100.0, raw=True)

def rolling_z(s, lookback, min_periods=120):
    """Trailing-window z-score of a series (no look-ahead)."""
    w = max(60, min(_win(lookback), max(len(s), 60)))
    mp = min(min_periods, max(30, w // 4))
    m = s.rolling(w, min_periods=mp).mean()
    sd = s.rolling(w, min_periods=mp).std()
    return (s - m) / sd.replace(0, np.nan)

# =============================================================================
# Vol data quality (data-driven, replaces opaque A/B/C)
# =============================================================================
def vol_quality(iv_series, window=60):
    """How 'live' the implied-vol series is: fraction of recent days it actually moves.

    Thresholds are calibrated to how commodity option marks actually behave on this
    feed — even very liquid names (WTI, Brent, gold) only print a fresh IV on ~50-70%
    of days, so the old 75%/50% cut-offs made nearly everything read Fair. Good >= 50%,
    Fair >= 35%, Weak below that. The stale flag still catches genuinely frozen marks.
    """
    s = iv_series.dropna()
    if len(s) < 25:
        return dict(tier="Weak", live=np.nan, stale=True, days=len(s))
    recent = s.tail(window)
    moves = recent.diff().abs().dropna()
    live = (moves > 0.01).mean() if len(moves) else 0.0
    # stale = barely moved at all recently
    stale = live < 0.35 or (recent.max() - recent.min()) < 0.25
    if live >= 0.50:
        tier = "Good"
    elif live >= 0.35:
        tier = "Fair"
    else:
        tier = "Weak"
    return dict(tier=tier, live=live, stale=bool(stale), days=len(s))

QUALITY_RANK = {"Good": 1, "Fair": 2, "Weak": 3}

# =============================================================================
# CONFIDENCE
# -----------------------------------------------------------------------------
# The confidence % is NOT a hand-tuned score. For every signal we ask one
# question of this asset's own 3y history:
#
#   "On past days when this exact setup was AT LEAST AS EXTREME as it is today,
#    how often did the thing we are betting on actually happen over the next
#    N days?"
#
# That raw hit rate is then (1) shrunk toward 50% by sample size, so a 4-for-4
# fluke does not read as 100%, and (2) pulled back toward 50% when the underlying
# IV data quality is Fair/Weak. Steps in detail:
#
#   raw   = wins / n                       empirical hit rate of comparable setups
#   p     = (wins + 10) / (n + 20)         Beta(10,10) posterior mean -> shrinkage
#   conf  = 50 + (p*100 - 50) * q          q = 1.0 Good, 0.85 Fair, 0.65 Weak
#   conf  = clip(conf, 20, 92)             never claim certainty in either direction
#
# Reading it: 50% = coin flip, this setup has no historical edge. >60% is a real
# edge worth sizing. <45% means the setup historically went the OTHER way — treat
# it as a warning, not a trade. n is always shown so you can see how much history
# stands behind the number.
# =============================================================================
CONF_PRIOR_W = 20                                        # pseudo-obs pulling toward 50%
QUALITY_FACTOR = {"Good": 1.0, "Fair": 0.85, "Weak": 0.65, "": 1.0}

def confidence(wins, n, tier=""):
    """-> (conf %, raw hit rate %, n). See the block comment above."""
    n = int(n)
    if n <= 0:
        return 50.0, np.nan, 0
    p = (wins + 0.5 * CONF_PRIOR_W) / (n + CONF_PRIOR_W)
    q = QUALITY_FACTOR.get(tier, 1.0)
    conf = 50.0 + (p * 100.0 - 50.0) * q
    return float(np.clip(conf, 20.0, 92.0)), float(wins) / n * 100.0, n

def conf_note(conf, raw, n, what, tier=""):
    """Plain-English explanation of where the confidence number came from."""
    if n < 5 or np.isnan(raw):
        return (f"Confidence {conf:.0f}% — only {n} comparable setups in 3y of history, "
                f"so this is held near a 50/50 prior. Treat as unproven, not as an edge.")
    qtxt = ""
    if tier in ("Fair", "Weak"):
        qtxt = (f" and pulled {'15' if tier == 'Fair' else '35'}% back toward 50% "
                f"because the IV data quality is {tier}")
    return (f"Confidence {conf:.0f}% — across {n} past setups at least this extreme, "
            f"{what} {raw:.0f}% of the time; shrunk toward 50% for sample size{qtxt}.")

def _count(cond, win):
    """(wins, n) over rows where the forward outcome is known."""
    cond = cond.fillna(False).astype(bool)
    n = int(cond.sum())
    if n == 0:
        return 0, 0
    return int((cond & win.fillna(False).astype(bool)).sum()), n

def hit_iv_reversion(s_iv, pr_s, cur_pr, horizon, cheap=True):
    """Cheap IV -> did IV rise over the next `horizon` days? Rich -> did it fall?"""
    fwd = s_iv.shift(-horizon) - s_iv
    ok = fwd.notna()
    if cheap:
        return _count((pr_s <= cur_pr) & ok, fwd > 0)
    return _count((pr_s >= cur_pr) & ok, fwd < 0)

def hit_vrp(s_iv, s_rv, sp_pr, cur_sp_pr, horizon, sell=True):
    """The real VRP test: did realized vol over the NEXT `horizon` days come in
    below (sell) / above (buy) the implied vol quoted on the day?"""
    fwd_rv = s_rv.shift(-horizon)          # rv window == horizon -> realized over [t, t+h]
    ok = fwd_rv.notna() & s_iv.notna()
    if sell:
        return _count((sp_pr >= cur_sp_pr) & ok, fwd_rv < s_iv)
    return _count((sp_pr <= cur_sp_pr) & ok, fwd_rv > s_iv)

def hit_disp_converge(sp, zs, z_now, horizon, lookback):
    """Did the vol spread move back toward its own mean over the next `horizon` days?"""
    w = max(60, min(_win(lookback), max(len(sp), 60)))
    m = sp.rolling(w, min_periods=min(120, max(30, w // 4))).mean()
    dev_now = (sp - m).abs()
    dev_fwd = (sp.shift(-horizon) - m).abs()
    ok = dev_fwd.notna() & dev_now.notna()
    same = np.sign(zs) == np.sign(z_now)
    return _count(same & (zs.abs() >= abs(z_now)) & ok, dev_fwd < dev_now)

def hit_corr_converge(close, a, b, zs, z_now, gap_now, s, horizon):
    """Did the a-vs-b performance gap close over the next `horizon` days?

    s = +1 when `a` is today's outperformer, -1 when `b` is. We look for past days
    with the same correlation regime, at least as stretched, and a gap at least as
    wide in the same direction — then check whether the gap actually narrowed.
    """
    la, lb = np.log(close[a].astype(float)), np.log(close[b].astype(float))
    G = (la - la.shift(horizon)) - (lb - lb.shift(horizon))     # trailing rel. perf
    F = (la.shift(-horizon) - la) - (lb.shift(-horizon) - lb)   # forward rel. perf
    ok = F.notna() & G.notna()
    same = (np.sign(zs) == np.sign(z_now)) & (zs.abs() >= abs(z_now))
    return _count(same & ((G * s) >= abs(gap_now)) & ok, (F * s) < 0)

def hit_leadlag(close, leader, foll, lag, gap_now):
    """Did the follower move the leader's way over the next `lag` days?"""
    L = close[leader].pct_change(lag)
    F = close[foll].pct_change(lag)
    gap = L - F
    fwd = close[foll].pct_change(lag).shift(-lag)
    s = 1.0 if gap_now >= 0 else -1.0
    ok = fwd.notna() & gap.notna()
    return _count(((gap * s) >= abs(gap_now) / 100.0) & ok, (fwd * s) > 0)

# =============================================================================
# Lead-lag
# =============================================================================
def lead_lag(x_ret, y_ret, max_lag=LEADLAG_MAX):
    best_k, best_c = 0, 0.0
    for k in range(-max_lag, max_lag + 1):
        if k == 0:
            continue
        c = x_ret.corr(y_ret.shift(-k))
        if pd.notna(c) and abs(c) > abs(best_c):
            best_k, best_c = k, c
    return best_k, best_c

def lead_lag_profile(x_ret, y_ret, max_lag=LEADLAG_MAX):
    """Full cross-correlation curve: corr(x_t, y_{t+k}) for k = -max_lag..+max_lag.
    Positive k with positive corr => x leads y by k days."""
    lags, corrs = [], []
    for k in range(-max_lag, max_lag + 1):
        lags.append(k)
        corrs.append(x_ret.corr(y_ret.shift(-k)) if k != 0 else x_ret.corr(y_ret))
    return lags, corrs

# =============================================================================
# Signal engines (set NAME / ASSET_CLASS before calling)
# =============================================================================

def _nm(t):
    return NAME.get(t, t)

def same_class(a, b):
    ca, cb = ASSET_CLASS.get(a), ASSET_CLASS.get(b)
    return ca is not None and ca == cb

def is_macro(t):
    return ASSET_CLASS.get(t) == MACRO_CLASS

def pair_allowed(a, b):
    """Correlation / lead-lag pairs: same asset class, OR anything vs a macro
    overlay (SPX / DXY / 10y). Cross-commodity-class pairs stay excluded — those
    were the noise (silver vs soybean meal)."""
    return same_class(a, b) or is_macro(a) or is_macro(b)

def build_signals(px, iv, cfg):
    """Return dict of per-engine DataFrames + supporting tables. cfg = dict of tunables."""
    close = px["close"]
    ret = close.pct_change()
    lookback = cfg["lookback"]
    rv = realized_vol(px, cfg["rv_estimator"], cfg["rv_window"])
    h_vol = CONF_HORIZON
    min_conf = cfg.get("min_conf", 0)

    # ---- quality table ----
    qrows = {}
    for tk in iv.columns:
        qrows[tk] = vol_quality(iv[tk])
    min_rank = QUALITY_RANK[cfg["min_quality"]]

    def vol_ok(tk):
        q = qrows.get(tk)
        if q is None:
            return False
        if QUALITY_RANK[q["tier"]] > min_rank:
            return False
        if cfg["exclude_stale"] and q["stale"]:
            return False
        return True

    iv_rows, e_ivmr, e_vrp = [], [], []
    for tk in iv.columns:
        nm, cls = _nm(tk), ASSET_CLASS.get(tk, "Other")
        q = qrows[tk]
        ivv, ivpr, ivz = pct_rank(iv[tk], lookback)
        if np.isnan(ivv):
            continue
        rv_last = rv[tk].dropna().iloc[-1] if tk in rv and rv[tk].dropna().size else np.nan
        spread = iv[tk] - rv[tk] if tk in rv else pd.Series(dtype=float)
        vrpv, vrppr, vrpz = pct_rank(spread, lookback) if len(spread.dropna()) else (np.nan, np.nan, np.nan)
        iv_rows.append(dict(name=nm, ticker=tk, cls=cls, tier=q["tier"], live=q["live"],
                            stale=q["stale"], iv=ivv, iv_pct=ivpr, rv=rv_last,
                            vrp=vrpv, vrp_pct=vrppr))
        if not vol_ok(tk):
            continue
        s_iv = iv[tk]
        pr_s = rolling_pct_rank(s_iv, lookback)
        pr_valid = pr_s.dropna()
        cur_pr = pr_valid.iloc[-1] if len(pr_valid) else np.nan

        # Engine 1: IV mean reversion
        if ivpr <= cfg["iv_lo"]:
            w, n = hit_iv_reversion(s_iv, pr_s, cur_pr, h_vol, cheap=True) if pd.notna(cur_pr) else (0, 0)
            conf, raw, n = confidence(w, n, q["tier"])
            e_ivmr.append(dict(score=(50 - ivpr) / 50, cls=cls, name=nm, side="BUY VOL",
                               conf=conf, conf_n=n, conf_raw=raw,
                               trigger=f"IV {ivv:.0f}, {ivpr:.0f}th pctile ({cfg['lb_name']})",
                               reason=(f"IV {ivv:.0f} sits in the bottom {ivpr:.0f}% of its {cfg['lb_name']} "
                                       f"range — options look cheap vs their own history. Buy vol "
                                       f"(long straddle/strangle or a debit spread) to be long gamma into a "
                                       f"likely vol pickup."), tier=q["tier"],
                               conf_note=conf_note(conf, raw, n,
                                                   f"implied vol was higher {h_vol} trading days later",
                                                   q["tier"])))
        elif ivpr >= cfg["iv_hi"]:
            w, n = hit_iv_reversion(s_iv, pr_s, cur_pr, h_vol, cheap=False) if pd.notna(cur_pr) else (0, 0)
            conf, raw, n = confidence(w, n, q["tier"])
            e_ivmr.append(dict(score=(ivpr - 50) / 50, cls=cls, name=nm, side="SELL VOL",
                               conf=conf, conf_n=n, conf_raw=raw,
                               trigger=f"IV {ivv:.0f}, {ivpr:.0f}th pctile ({cfg['lb_name']})",
                               reason=(f"IV {ivv:.0f} sits in the top {100-ivpr:.0f}% of its {cfg['lb_name']} "
                                       f"range — options look rich vs their own history. Sell vol "
                                       f"(credit spread / iron condor) to collect premium into a likely vol fade."),
                               tier=q["tier"],
                               conf_note=conf_note(conf, raw, n,
                                                   f"implied vol was lower {h_vol} trading days later",
                                                   q["tier"])))

        # Engine 2: Variance risk premium (IV vs realized)
        if not np.isnan(vrppr) and len(spread.dropna()):
            sp_pr = rolling_pct_rank(spread, lookback)
            sp_valid = sp_pr.dropna()
            cur_sp_pr = sp_valid.iloc[-1] if len(sp_valid) else np.nan
            hz = int(cfg["rv_window"])
            if vrppr >= cfg["iv_hi"]:
                w, n = hit_vrp(s_iv, rv[tk], sp_pr, cur_sp_pr, hz, sell=True) if pd.notna(cur_sp_pr) else (0, 0)
                conf, raw, n = confidence(w, n, q["tier"])
                e_vrp.append(dict(score=(vrppr - 50) / 50, cls=cls, name=nm, side="SELL VOL",
                                  conf=conf, conf_n=n, conf_raw=raw,
                                  trigger=f"IV-RV {vrpv:+.0f} pts, {vrppr:.0f}th pctile",
                                  reason=(f"IV {ivv:.0f} vs realized {rv_last:.0f} → the market is pricing "
                                          f"{vrpv:+.0f} vol pts MORE than {nm} is actually delivering, wide vs "
                                          f"its own history. Sell vol to harvest that premium; the option seller "
                                          f"wins if realized stays below implied."), tier=q["tier"],
                                  conf_note=conf_note(conf, raw, n,
                                                      f"realized vol over the following {hz}d came in BELOW "
                                                      f"the implied vol quoted that day", q["tier"])))
            elif vrppr <= cfg["iv_lo"]:
                w, n = hit_vrp(s_iv, rv[tk], sp_pr, cur_sp_pr, hz, sell=False) if pd.notna(cur_sp_pr) else (0, 0)
                conf, raw, n = confidence(w, n, q["tier"])
                e_vrp.append(dict(score=(50 - vrppr) / 50, cls=cls, name=nm, side="BUY VOL",
                                  conf=conf, conf_n=n, conf_raw=raw,
                                  trigger=f"IV-RV {vrpv:+.0f} pts, {vrppr:.0f}th pctile",
                                  reason=(f"IV {ivv:.0f} vs realized {rv_last:.0f} → the market is pricing barely "
                                          f"more (or less) vol than {nm} is delivering, low vs its own history. "
                                          f"Buy vol — cheap optionality if realized keeps up with or exceeds implied."),
                                  tier=q["tier"],
                                  conf_note=conf_note(conf, raw, n,
                                                      f"realized vol over the following {hz}d came in ABOVE "
                                                      f"the implied vol quoted that day", q["tier"])))

    iv_table = pd.DataFrame(iv_rows)

    # ---- vol-spread dispersion (same class, both legs need live IV) ----
    ivcols = [c for c in iv.columns if vol_ok(c)]
    e_disp = []
    for i in range(len(ivcols)):
        for j in range(i + 1, len(ivcols)):
            a, b = ivcols[i], ivcols[j]
            if not same_class(a, b):
                continue
            sp = (iv[a] - iv[b])
            cur, mean, sd, z = band_z(sp, lookback)
            if np.isnan(z) or abs(z) < cfg["disp_z"]:
                continue
            iva = iv[a].dropna().iloc[-1]; ivb = iv[b].dropna().iloc[-1]
            rich, cheap = (a, b) if z > 0 else (b, a)
            zs = rolling_z(sp, lookback)
            w, n = hit_disp_converge(sp, zs, z, h_vol, lookback)
            worst = "Weak" if "Weak" in (qrows[a]["tier"], qrows[b]["tier"]) else (
                    "Fair" if "Fair" in (qrows[a]["tier"], qrows[b]["tier"]) else "Good")
            conf, raw, n = confidence(w, n, worst)
            e_disp.append(dict(score=min(abs(z) / 3, 1.5), cls=ASSET_CLASS.get(a), name=f"{_nm(a)} / {_nm(b)}",
                               conf=conf, conf_n=n, conf_raw=raw,
                               side=f"SELL {_nm(rich)} VOL / BUY {_nm(cheap)} VOL",
                               trigger=f"IV spread {cur:+.0f} pts, {z:+.1f}σ vs {cfg['lb_name']} band",
                               reason=(f"{_nm(a)} IV {iva:.0f} vs {_nm(b)} IV {ivb:.0f}: the spread ({cur:+.0f} pts) is "
                                       f"{abs(z):.1f}σ from its normal band (avg {mean:+.0f} pts). "
                                       f"These two normally track, so expect the spread to revert — sell the "
                                       f"relatively expensive vol ({_nm(rich)}), buy the cheap one ({_nm(cheap)})."),
                               tier=worst,
                               conf_note=conf_note(conf, raw, n,
                                                   f"the spread moved back toward its mean over the next {h_vol}d",
                                                   worst)))

    # ---- correlation RV & lead-lag (same class, or anything vs a macro overlay) ----
    pxcols = [c for c in close.columns if c in ret.columns]
    e_corr, e_ll = [], []
    for i in range(len(pxcols)):
        for j in range(i + 1, len(pxcols)):
            a, b = pxcols[i], pxcols[j]
            if not pair_allowed(a, b):
                continue
            na, nb = _nm(a), _nm(b)
            macro_pair = is_macro(a) or is_macro(b)
            min_mean = MIN_MEAN_CORR_MACRO if macro_pair else MIN_MEAN_CORR

            # ----- Engine 4: Correlation RV (a CONVERGENCE / relative-value trade) -----
            # Two normally-correlated names. When their 60d correlation is stretched
            # from its own trailing norm (either direction) we fade the RELATIVE-
            # PERFORMANCE GAP: BUY the underperformer, SELL the outperformer, betting
            # the pair re-converges. Mean-reversion on the spread — deliberately NOT
            # the same as lead-lag catch-up below, which follows the leader's direction.
            rcr = ret[a].rolling(cfg["corr_window"]).corr(ret[b]).dropna()
            if len(rcr) >= 150:
                zs = rolling_z(rcr, lookback)
                zz = zs.dropna()
                rm = rcr.rolling(max(60, min(_win(lookback), len(rcr))),
                                 min_periods=120).mean().dropna()
                if len(zz) and len(rm):
                    z, cur, mean = zz.iloc[-1], rcr.iloc[-1], rm.iloc[-1]
                    if pd.notna(z) and abs(mean) >= min_mean and abs(z) >= cfg["corr_z"]:
                        ra = recent_return(close, a, cfg["pair_win"])
                        rb = recent_return(close, b, cfg["pair_win"])
                        if pd.notna(ra) and pd.notna(rb):
                            # out = better recent performer, und = laggard.
                            # Convergence -> buy und, sell out.
                            out, und = (a, b) if ra >= rb else (b, a)
                            r_out, r_und = (ra, rb) if ra >= rb else (rb, ra)
                            gap = r_out - r_und                      # >= 0, in %
                            s_dir = 1.0 if out == a else -1.0
                            w, n = hit_corr_converge(close, a, b, zs, z,
                                                     gap / 100.0, s_dir, cfg["pair_win"])
                            conf, raw, n = confidence(w, n)
                            if z < 0:
                                regime = (f"{na} and {nb} normally move together (avg corr {mean:+.2f}), but "
                                          f"their 60d correlation has dropped to {cur:+.2f} ({abs(z):.1f}σ below "
                                          f"normal) — they've decoupled and a gap has opened.")
                            else:
                                regime = (f"{na} and {nb} are moving in unusually tight lockstep right now "
                                          f"(corr {cur:+.2f} vs avg {mean:+.2f}, {abs(z):.1f}σ above normal) — a "
                                          f"strong common-factor regime, so any gap between them should be short-lived.")
                            xtag = " Cross-asset macro pair." if macro_pair else ""
                            e_corr.append(dict(score=min(abs(z)/3, 1.5), cls=ASSET_CLASS.get(a),
                                name=f"{na} / {nb}", side=f"BUY {_nm(und)} / SELL {_nm(out)}",
                                conf=conf, conf_n=n, conf_raw=raw,
                                trigger=f"60d corr {cur:+.2f} vs avg {mean:+.2f} ({z:+.1f}σ), gap {gap:.1f}%",
                                reason=(f"{regime} Over the last {cfg['pair_win']}d {_nm(out)} is {r_out:+.1f}% "
                                        f"vs {_nm(und)} {r_und:+.1f}% — a {gap:.1f}% gap. This is a convergence "
                                        f"(relative-value) trade, NOT a directional momentum bet: BUY the "
                                        f"underperformer {_nm(und)}, SELL the outperformer {_nm(out)}, expecting "
                                        f"the gap to close as the relationship normalises.{xtag}"), tier="",
                                conf_note=conf_note(conf, raw, n,
                                                    f"the gap closed over the next {cfg['pair_win']}d")))

            # ----- Engine 5: Lead-lag catch-up (directional follow-through) -----
            win = ret[[a, b]].tail(cfg["ll_window"]).dropna()
            if len(win) >= 40:
                k, pk = lead_lag(win[a], win[b])
                if k != 0 and abs(pk) >= cfg["ll_r"]:
                    leader, foll = (a, b) if k > 0 else (b, a)
                    lag = abs(k); nL, nF = _nm(leader), _nm(foll)
                    lm = recent_return(close, leader, lag); fm = recent_return(close, foll, lag)
                    if pd.notna(lm) and pd.notna(fm):
                        gap = lm - fm
                        if abs(gap) >= cfg["ll_gap"]:
                            side = "BUY" if gap > 0 else "SELL"
                            w, n = hit_leadlag(close, leader, foll, lag, gap)
                            conf, raw, n = confidence(w, n)
                            xtag = " Cross-asset macro pair." if macro_pair else ""
                            e_ll.append(dict(score=min(abs(gap)/max(cfg["ll_gap"]*2, .01), 1.4),
                                cls=ASSET_CLASS.get(a), name=f"{nL} leads {nF}", side=f"{side} {nF}",
                                conf=conf, conf_n=n, conf_raw=raw,
                                trigger=f"{lag}d lead, peak r {pk:+.2f}, gap {gap:+.1f}%",
                                reason=(f"Over the last {cfg['ll_window']}d, {nL} moves ~{lag}d ahead of {nF} "
                                        f"(peak lead-lag r {pk:+.2f}). {nL} has already moved {lm:+.1f}% but {nF} "
                                        f"only {fm:+.1f}% — a {gap:+.1f}% gap. Unlike the convergence trade, here "
                                        f"you follow the leader's DIRECTION: expect {nF} to catch up the same way, "
                                        f"so {side} {nF}.{xtag}"),
                                tier="",
                                conf_note=conf_note(conf, raw, n,
                                                    f"{nF} went on to move the same way as {nL} over the next {lag}d")))

    def mk(rows):
        if not rows:
            return pd.DataFrame()
        df = pd.DataFrame(rows)
        if min_conf:
            df = df[df["conf"] >= min_conf]
        if df.empty:
            return df
        return df.sort_values(["conf", "score"], ascending=[False, False])

    return dict(px=px, iv=iv, rv=rv, ret=ret, iv_table=iv_table, quality=qrows,
                engines={"IV mean-reversion": mk(e_ivmr), "Variance risk premium": mk(e_vrp),
                         "Vol dispersion (pairs)": mk(e_disp), "Correlation RV": mk(e_corr),
                         "Lead-lag catch-up": mk(e_ll)})

In [ ]:
# =====================================================================
# DATA LAYER — pull OHLC + implied vol from Bloomberg (BQL)
# =====================================================================
def _pull(tickers, item_fn, start, end):
    field = item_fn(dates=bq.func.range(start, end))
    frames, failed = {}, []
    for tk in tickers:
        try:
            res = bq.execute(bql.Request(tk, field))
            d = res[0].df().reset_index()
            d = d.drop(columns=[c for c in ["CURRENCY"] if c in d.columns])
            vcols = [c for c in d.columns if c not in ["ID", "DATE"]]
            if not vcols:
                failed.append(tk); continue
            s = d.set_index("DATE")[vcols[0]].sort_index()
            s = s[~s.index.duplicated(keep="last")]
            if s.dropna().empty:
                failed.append(tk)
            else:
                frames[tk] = s
        except Exception:
            failed.append(tk)
    df = pd.DataFrame(frames).sort_index() if frames else pd.DataFrame()
    return df, failed


def fetch_all(status=None):
    end = pd.Timestamp("today").strftime("%Y-%m-%d")
    start = (pd.Timestamp("today") - pd.DateOffset(years=HISTORY_YEARS)).strftime("%Y-%m-%d")

    def note(m):
        if status is not None:
            status.value = "<span style='color:%s'>%s</span>" % (MUTED, m)

    note("Pulling prices (open/high/low/close)…")
    o, _ = _pull(ALL_TICKERS, bq.data.px_open, start, end)
    h, _ = _pull(ALL_TICKERS, bq.data.px_high, start, end)
    l, _ = _pull(ALL_TICKERS, bq.data.px_low, start, end)
    c, px_fail = _pull(ALL_TICKERS, bq.data.px_last, start, end)

    close = c.ffill()
    idx = close.index

    def prep(df):
        # Align to the close grid; fall back to close where OHLC is missing so
        # range-based estimators degrade gracefully instead of crashing.
        if df.empty:
            return close.copy()
        df = df.reindex(index=idx).ffill()
        for tk in close.columns:
            if tk not in df.columns or df[tk].dropna().empty:
                df[tk] = close[tk]
        return df[close.columns]

    px = {"open": prep(o), "high": prep(h), "low": prep(l), "close": close}

    note("Pulling implied vol…")
    iv, iv_fail = _pull([t for t in IV_TICKERS if t in close.columns],
                        bq.data.hist_put_imp_vol, start, end)
    iv = iv.ffill()
    return px, iv, px_fail, iv_fail

In [ ]:
# =====================================================================
# RENDERERS — signal tables, vol table, drill-down charts
# =====================================================================
def grouped_options(tickers):
    """Dropdown options grouped by asset class with non-selectable headers."""
    opts, by = [], {}
    for t in tickers:
        by.setdefault(ASSET_CLASS.get(t, "Other"), []).append(t)
    for cls in CLASS_ORDER + ["Other"]:
        if cls in by:
            opts.append(("──  %s  ──" % cls.upper(), None))
            for t in sorted(by[cls], key=lambda x: NAME.get(x, x)):
                opts.append(("    " + NAME.get(t, t), t))
    return opts


def _action_color(side):
    s = side.upper()
    has_buy, has_sell = "BUY" in s, "SELL" in s
    if has_buy and not has_sell:
        return GREEN
    if has_sell and not has_buy:
        return RED
    return TXT


def _conf_color(c):
    """>=65 strong, 55-65 modest, 45-55 coin-flip, <45 historically adverse."""
    if c is None or (isinstance(c, float) and np.isnan(c)):
        return MUTED
    if c >= 65:
        return GREEN
    if c >= 55:
        return TEAL
    if c >= 45:
        return AMBER
    return RED


def render_engine(title, accent, df):
    if df is None or df.empty:
        return ("<div style='margin:6px 0 18px'>"
                "<div style='font:700 14px Inter,Arial;color:%s'>%s</div>"
                "<div style='color:%s;font:400 12px Inter,Arial;padding:8px 2px'>"
                "No signals at the current settings.</div></div>" % (accent, title, MUTED))
    rows = ""
    for _, r in df.iterrows():
        ac = _action_color(r["side"])
        conf = r.get("conf", np.nan)
        cc = _conf_color(conf)
        n = int(r.get("conf_n", 0) or 0)
        note = r.get("conf_note", "")
        rows += (
            "<tr style='border-top:1px solid %s'>"
            "<td style='padding:9px 12px;font:600 13px Inter,Arial;color:%s;white-space:nowrap'>%s</td>"
            "<td style='padding:9px 12px;font:800 12px Inter,Arial;color:%s;white-space:nowrap'>%s</td>"
            "<td style='padding:9px 12px;text-align:center;white-space:nowrap'>"
            "<div style='font:800 15px Inter,Arial;color:%s'>%.0f%%</div>"
            "<div style='font:400 10px Inter,Arial;color:%s'>n=%d</div></td>"
            "<td style='padding:9px 12px;font:500 12px Inter,Arial;color:%s;white-space:nowrap'>%s</td>"
            "<td style='padding:9px 12px;font:400 12px Inter,Arial;color:%s;line-height:1.45'>%s"
            "<div style='margin-top:5px;font:400 11px Inter,Arial;color:%s;font-style:italic'>%s</div>"
            "</td></tr>" % (
                GRID, TXT, r["name"], ac, r["side"],
                cc, conf if pd.notna(conf) else 50.0, MUTED, n,
                MUTED, r["trigger"], TXT, r["reason"], MUTED, note)
        )
    n = len(df)
    return (
        "<div style='margin:6px 0 20px'>"
        "<div style='display:flex;align-items:center;gap:8px;margin-bottom:6px'>"
        "<span style='width:9px;height:9px;border-radius:2px;background:%s;display:inline-block'></span>"
        "<span style='font:700 14px Inter,Arial;color:%s'>%s</span>"
        "<span style='font:600 11px Inter,Arial;color:%s'>&nbsp;%d</span></div>"
        "<table style='border-collapse:collapse;width:100%%;background:%s;border:1px solid %s;"
        "border-radius:10px;overflow:hidden'>"
        "<tr style='background:#1B2230'>"
        "<th style='text-align:left;padding:8px 12px;font:700 10px Inter,Arial;letter-spacing:.08em;color:%s'>ASSET / PAIR</th>"
        "<th style='text-align:left;padding:8px 12px;font:700 10px Inter,Arial;letter-spacing:.08em;color:%s'>ACTION</th>"
        "<th style='text-align:center;padding:8px 12px;font:700 10px Inter,Arial;letter-spacing:.08em;color:%s'>CONFIDENCE</th>"
        "<th style='text-align:left;padding:8px 12px;font:700 10px Inter,Arial;letter-spacing:.08em;color:%s'>TRIGGER</th>"
        "<th style='text-align:left;padding:8px 12px;font:700 10px Inter,Arial;letter-spacing:.08em;color:%s'>WHY / WHAT TO DO</th>"
        "</tr>%s</table></div>"
        % (accent, TXT, title, MUTED, n, PANEL, GRID, MUTED, MUTED, MUTED, MUTED, MUTED, rows)
    )


def render_signals(res):
    blocks = [render_engine(name, clr, res["engines"].get(name)) for name, clr in ENGINE_ORDER]
    total = sum(len(res["engines"].get(n)) for n, _ in ENGINE_ORDER)
    head = ("<div style='font:400 12px Inter,Arial;color:%s;margin:0 0 10px'>"
            "%d signals across %d engines, sorted by confidence. Confidence = how often this "
            "same setup, at least this extreme, actually worked over the next N days in 3y of "
            "this asset's own history (shrunk toward 50%% by sample size <b>n</b>). "
            "50%% = coin flip &nbsp;·&nbsp; &gt;60%% = real edge &nbsp;·&nbsp; &lt;45%% = historically "
            "went the other way.</div>" % (MUTED, total, len(ENGINE_ORDER)))
    return widgets.HTML(head + "".join(blocks))


def render_iv_table(res):
    t = res["iv_table"].copy()
    if t.empty:
        return widgets.HTML("<div style='color:%s'>No implied-vol data.</div>" % MUTED)
    t = t.sort_values(["stale", "cls", "iv_pct"], ascending=[True, True, False])

    def heat(p):
        if p is None or (isinstance(p, float) and np.isnan(p)):
            return MUTED
        return RED if p >= 80 else GREEN if p <= 20 else TXT

    tier_clr = {"Good": GREEN, "Fair": AMBER, "Weak": RED}
    rows = ""
    for _, r in t.iterrows():
        live = "" if (r["live"] is None or (isinstance(r["live"], float) and np.isnan(r["live"]))) else "%.0f%%" % (r["live"] * 100)
        stale_txt = "STALE" if r["stale"] else ""
        rows += (
            "<tr style='border-top:1px solid %s'>"
            "<td style='padding:7px 12px;font:600 12px Inter,Arial;color:%s'>%s</td>"
            "<td style='padding:7px 12px;color:%s'>%s</td>"
            "<td style='padding:7px 12px;text-align:center;font:700 11px Inter,Arial;color:%s'>%s</td>"
            "<td style='padding:7px 12px;text-align:center;color:%s'>%s</td>"
            "<td style='padding:7px 12px;text-align:center;color:%s'>%s</td>"
            "<td style='padding:7px 12px;text-align:right;color:%s'>%.0f</td>"
            "<td style='padding:7px 12px;text-align:right;font:700 12px Inter,Arial;color:%s'>%.0f</td>"
            "<td style='padding:7px 12px;text-align:right;color:%s'>%.0f</td>"
            "<td style='padding:7px 12px;text-align:right;color:%s'>%+.0f</td>"
            "<td style='padding:7px 12px;text-align:right;font:700 12px Inter,Arial;color:%s'>%s</td>"
            "</tr>" % (
                GRID, TXT, r["name"], MUTED, r["cls"],
                tier_clr.get(r["tier"], MUTED), r["tier"],
                MUTED, live, AMBER if r["stale"] else MUTED, stale_txt,
                TXT, r["iv"], heat(r["iv_pct"]), r["iv_pct"],
                TXT, r["rv"], TXT, r["vrp"],
                heat(r["vrp_pct"]), ("%.0f" % r["vrp_pct"]) if not np.isnan(r["vrp_pct"]) else "-",
            )
        )
    heads = [("ASSET", "left"), ("CLASS", "left"), ("QUALITY", "center"),
             ("LIVE", "center"), ("STALE", "center"), ("IV", "right"),
             ("IV PCTILE", "right"), ("REALIZED", "right"),
             ("IV−RV", "right"), ("IV−RV PCTILE", "right")]
    th = "".join("<th style='padding:8px 12px;text-align:%s;font:700 10px Inter,Arial;"
                 "letter-spacing:.06em;color:%s'>%s</th>" % (al, MUTED, h) for h, al in heads)
    return widgets.HTML(
        "<table style='border-collapse:collapse;width:100%%;background:%s;border:1px solid %s;"
        "border-radius:10px;overflow:hidden'><tr style='background:#1B2230'>%s</tr>%s</table>"
        % (PANEL, GRID, th, rows)
    )


def iv_drill_fig(res, tk):
    iv, rv = res["iv"], res["rv"]
    s_iv = iv[tk].dropna()
    s_rv = rv[tk].reindex(s_iv.index) if tk in rv else pd.Series(index=s_iv.index, dtype=float)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=s_iv.index, y=s_iv, name="Implied vol",
                             line=dict(color=BLUE, width=2.4)))
    fig.add_trace(go.Scatter(x=s_rv.index, y=s_rv, name="Realized vol",
                             line=dict(color=AMBER, width=1.8, dash="dot")))
    for q, lbl in [(0.1, "IV 10th"), (0.9, "IV 90th")]:
        fig.add_hline(y=s_iv.quantile(q), line=dict(color=MUTED, width=1, dash="dash"),
                      annotation_text=lbl, annotation_font_color=MUTED)
    fig.update_layout(template="plotly_dark", height=430, paper_bgcolor=BG, plot_bgcolor=BG,
                      title="<b>%s — Implied vs Realized Vol</b>" % NAME.get(tk, tk),
                      font=dict(family="Inter", color=TXT), hovermode="x unified",
                      legend=dict(orientation="h", y=1.03, x=1, xanchor="right"),
                      margin=dict(l=55, r=20, t=60, b=35),
                      xaxis=dict(gridcolor=GRID), yaxis=dict(title="vol %", gridcolor=GRID))
    return go.FigureWidget(fig)


def corr_pair_fig(res, a, b):
    ret = res["ret"]
    rcr = ret[a].rolling(CORR_WINDOW).corr(ret[b]).dropna()
    if rcr.empty:
        return widgets.HTML("<div style='color:%s'>No overlapping data for this pair.</div>" % MUTED)
    mean, sd = rcr.mean(), rcr.std()
    win = ret[[a, b]].tail(LEADLAG_WINDOW).dropna()
    k, pk = lead_lag(win[a], win[b]) if len(win) >= 40 else (0, np.nan)
    na, nb = NAME.get(a, a), NAME.get(b, b)
    if k > 0:
        ll = "%s leads %s by %dd, peak r %+.2f" % (na, nb, k, pk)
    elif k < 0:
        ll = "%s leads %s by %dd, peak r %+.2f" % (nb, na, abs(k), pk)
    else:
        ll = "no clear lead-lag"
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=rcr.index, y=rcr, name="%dd price corr" % CORR_WINDOW,
                             line=dict(color=TEAL, width=2.4)))
    fig.add_hline(y=mean, line=dict(color=MUTED, width=1), annotation_text="avg",
                  annotation_font_color=MUTED)
    if pd.notna(sd):
        for s, lbl in [(2, "+2σ"), (-2, "−2σ")]:
            fig.add_hline(y=mean + s * sd, line=dict(color=GRID, width=1, dash="dash"),
                          annotation_text=lbl, annotation_font_color=MUTED)
    fig.update_layout(template="plotly_dark", height=430, paper_bgcolor=BG, plot_bgcolor=BG,
                      title="<b>%s / %s — Price-return correlation</b>  ·  %s" % (na, nb, ll),
                      font=dict(family="Inter", color=TXT), hovermode="x unified",
                      margin=dict(l=55, r=20, t=60, b=35),
                      xaxis=dict(gridcolor=GRID), yaxis=dict(title="correlation", gridcolor=GRID))
    return go.FigureWidget(fig)


def vol_corr_pair_fig(res, a, b):
    """Rolling correlation of the two assets' *implied vols* (day-over-day IV changes).

    Answers "do these two names' vols move together?" — distinct from the price-return
    correlation above. High vol-correlation = their option markets re-price risk in
    tandem (good backdrop for a vol-dispersion pairs trade); a breakdown flags one vol
    surface moving on its own.
    """
    iv = res["iv"]
    na, nb = NAME.get(a, a), NAME.get(b, b)
    if a not in iv.columns or b not in iv.columns:
        return widgets.HTML(
            "<div style='color:%s;font:400 12px Inter,Arial;padding:10px 2px'>"
            "Vol correlation needs a listed implied-vol series on both legs — not available "
            "for this pair (one or both are price-only: LME / SGX / macro names).</div>" % MUTED)
    dv = pd.concat([iv[a].diff(), iv[b].diff()], axis=1).dropna()
    if len(dv) < CORR_WINDOW + 5:
        return widgets.HTML("<div style='color:%s'>Not enough overlapping vol data for this pair.</div>" % MUTED)
    vcr = dv.iloc[:, 0].rolling(CORR_WINDOW).corr(dv.iloc[:, 1]).dropna()
    if vcr.empty:
        return widgets.HTML("<div style='color:%s'>No overlapping vol data for this pair.</div>" % MUTED)
    mean, sd, cur = vcr.mean(), vcr.std(), vcr.iloc[-1]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=vcr.index, y=vcr, name="%dd vol corr" % CORR_WINDOW,
                             line=dict(color=PURPLE, width=2.4)))
    fig.add_hline(y=mean, line=dict(color=MUTED, width=1), annotation_text="avg",
                  annotation_font_color=MUTED)
    if pd.notna(sd):
        for s, lbl in [(2, "+2σ"), (-2, "−2σ")]:
            fig.add_hline(y=mean + s * sd, line=dict(color=GRID, width=1, dash="dash"),
                          annotation_text=lbl, annotation_font_color=MUTED)
    fig.update_layout(template="plotly_dark", height=430, paper_bgcolor=BG, plot_bgcolor=BG,
                      title="<b>%s / %s — Implied-vol correlation (ΔIV)</b>  ·  now %+.2f, avg %+.2f"
                            % (na, nb, cur, mean),
                      font=dict(family="Inter", color=TXT), hovermode="x unified",
                      margin=dict(l=55, r=20, t=60, b=35),
                      xaxis=dict(gridcolor=GRID),
                      yaxis=dict(title="corr of daily IV changes", gridcolor=GRID))
    return go.FigureWidget(fig)


def leadlag_pair_fig(res, a, b):
    """Make the lead-lag visible instead of just asserting it.

    Top panel  — both prices rebased to 100, PLUS the leader re-drawn shifted
                 forward by its measured lead. If the lead is real, that dashed
                 shifted line sits on top of the follower: you can see the leader's
                 turns arriving in the follower N days later.
    Bottom panel — the cross-correlation curve corr(x_t, y_{t+k}) for every lag k.
                 The tallest bar is the lead. Bars right of zero = A leads B; bars
                 left of zero = B leads A. A flat curve means there is no real
                 timing edge, only contemporaneous correlation.
    """
    close, ret = res["px"]["close"], res["ret"]
    na, nb = NAME.get(a, a), NAME.get(b, b)
    win = ret[[a, b]].tail(LEADLAG_WINDOW).dropna()
    if len(win) < 40:
        return widgets.HTML("<div style='color:%s'>Not enough overlapping data for lead-lag.</div>" % MUTED)

    k, pk = lead_lag(win[a], win[b])
    lags, corrs = lead_lag_profile(win[a], win[b])
    if k > 0:
        leader, foll, lag = a, b, k
    elif k < 0:
        leader, foll, lag = b, a, -k
    else:
        leader, foll, lag = a, b, 0
    nL, nF = NAME.get(leader, leader), NAME.get(foll, foll)

    disp = close[[leader, foll]].tail(LEADLAG_WINDOW + max(lag, 1)).dropna()
    if disp.empty:
        return widgets.HTML("<div style='color:%s'>No overlapping prices for this pair.</div>" % MUTED)
    rb = disp / disp.iloc[0] * 100.0
    shifted = rb[leader].shift(lag) if lag else None

    fig = make_subplots(rows=2, cols=1, row_heights=[0.62, 0.38], vertical_spacing=0.13,
                        subplot_titles=("Rebased prices (=100 at window start)",
                                        "Cross-correlation by lag — tallest bar is the lead"))
    fig.add_trace(go.Scatter(x=rb.index, y=rb[leader], name="%s (leader)" % nL,
                             line=dict(color=BLUE, width=2.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=rb.index, y=rb[foll], name="%s (follower)" % nF,
                             line=dict(color=AMBER, width=2.2)), row=1, col=1)
    if lag:
        fig.add_trace(go.Scatter(x=shifted.index, y=shifted,
                                 name="%s shifted +%dd" % (nL, lag),
                                 line=dict(color=GREEN, width=2.0, dash="dash")), row=1, col=1)

    bar_clr = [PURPLE if L == k else TEAL for L in lags]
    fig.add_trace(go.Bar(x=lags, y=corrs, marker_color=bar_clr, name="corr by lag",
                         showlegend=False,
                         hovertemplate="lag %{x}d · r %{y:.2f}<extra></extra>"), row=2, col=1)
    fig.add_vline(x=0, line=dict(color=MUTED, width=1), row=2, col=1)

    if lag:
        headline = ("<b>%s leads %s by %dd</b>  ·  peak r %+.2f  ·  green dashed = %s shifted "
                    "forward %dd (should overlay %s)" % (nL, nF, lag, pk, nL, lag, nF))
    else:
        headline = "<b>%s / %s</b>  ·  no clear lead-lag at any tested lag" % (na, nb)

    fig.update_layout(template="plotly_dark", height=640, paper_bgcolor=BG, plot_bgcolor=BG,
                      title=headline, font=dict(family="Inter", color=TXT),
                      hovermode="x unified", bargap=0.35,
                      legend=dict(orientation="h", y=1.06, x=1, xanchor="right"),
                      margin=dict(l=55, r=20, t=95, b=40))
    fig.update_xaxes(gridcolor=GRID, row=1, col=1)
    fig.update_xaxes(gridcolor=GRID, row=2, col=1,
                     title="lag k (days) — positive: %s leads %s   |   negative: %s leads %s"
                           % (na, nb, nb, na),
                     dtick=2)
    fig.update_yaxes(gridcolor=GRID, title="rebased", row=1, col=1)
    fig.update_yaxes(gridcolor=GRID, title="corr(%s_t, %s_t+k)" % (na, nb), row=2, col=1)
    for ann in fig.layout.annotations:
        ann.font.size = 12
        ann.font.color = MUTED
    return go.FigureWidget(fig)

In [ ]:
# =====================================================================
# CONTROLS + LAYOUT + WIRING
# =====================================================================
LBL = {"description_width": "150px"}
WIDE = widgets.Layout(width="330px")

lookback_dd = widgets.Dropdown(options=[("1 year", 252), ("2 years", 504), ("3 years", 0)],
                               value=252, description="Percentile lookback:", style=LBL, layout=WIDE)
rv_est_dd = widgets.Dropdown(options=list(RV_ESTIMATORS.keys()), value="Yang-Zhang (OHLC)",
                             description="Realized-vol estimator:", style=LBL, layout=WIDE)
rv_win_dd = widgets.Dropdown(options=[("1 week", 5), ("1 month", 21), ("3 months", 63)],
                             value=21, description="Realized-vol window:", style=LBL, layout=WIDE)
iv_lo = widgets.IntSlider(value=10, min=5, max=30, step=5, description="Cheap ≤ pctile:",
                          continuous_update=False, style=LBL, layout=WIDE)
iv_hi = widgets.IntSlider(value=90, min=70, max=95, step=5, description="Rich ≥ pctile:",
                          continuous_update=False, style=LBL, layout=WIDE)
disp_z = widgets.FloatSlider(value=2.0, min=1.0, max=3.5, step=0.25, description="Vol-spread |z| ≥:",
                             continuous_update=False, style=LBL, layout=WIDE)
corr_z = widgets.FloatSlider(value=1.0, min=1.0, max=3.5, step=0.25, description="Corr break |z| ≥:",
                             continuous_update=False, style=LBL, layout=WIDE)
ll_r = widgets.FloatSlider(value=0.30, min=0.30, max=0.90, step=0.05, description="Lead-lag r ≥:",
                           continuous_update=False, style=LBL, layout=WIDE)
ll_gap = widgets.FloatSlider(value=1.5, min=0.5, max=8.0, step=0.5, description="Lead-lag gap % ≥:",
                             continuous_update=False, style=LBL, layout=WIDE)
min_conf = widgets.IntSlider(value=0, min=0, max=80, step=5, description="Min confidence %:",
                             continuous_update=False, style=LBL, layout=WIDE)
min_quality = widgets.Dropdown(options=[("Good only", "Good"), ("Good + Fair", "Fair"),
                                        ("All (incl. Weak)", "Weak")],
                               value="Fair", description="Min vol quality:", style=LBL, layout=WIDE)
exclude_stale = widgets.Checkbox(value=True, description="Exclude stale IV", indent=False,
                                 layout=widgets.Layout(width="200px"))

load_btn = widgets.Button(description="Load / refresh data", button_style="success",
                          layout=widgets.Layout(width="200px", height="36px"))
status = widgets.HTML()

iv_drill_dd = widgets.Dropdown(description="Asset:", style=LBL, layout=WIDE)
pair_a_dd = widgets.Dropdown(description="Asset A:", style=LBL, layout=WIDE)
pair_b_dd = widgets.Dropdown(description="Asset B:", style=LBL, layout=WIDE)

signals_box = widgets.VBox()
iv_table_box = widgets.VBox()
iv_chart_box = widgets.VBox()
pair_chart_box = widgets.VBox()
vol_pair_chart_box = widgets.VBox()
leadlag_chart_box = widgets.VBox()
disloc_box = widgets.VBox()


def current_cfg():
    lb_name = {252: "1y", 504: "2y", 0: "3y"}[lookback_dd.value]
    return dict(lookback=lookback_dd.value, lb_name=lb_name,
                rv_estimator=rv_est_dd.value, rv_window=rv_win_dd.value,
                iv_lo=iv_lo.value, iv_hi=iv_hi.value,
                disp_z=disp_z.value, corr_z=corr_z.value,
                corr_window=CORR_WINDOW, pair_win=PAIR_WIN,
                ll_window=LEADLAG_WINDOW, ll_r=ll_r.value, ll_gap=ll_gap.value,
                min_quality=min_quality.value, exclude_stale=exclude_stale.value,
                min_conf=min_conf.value)


def recompute(*_):
    if "px" not in STATE:
        status.value = "<span style='color:%s'>Click “Load / refresh data” first.</span>" % AMBER
        return
    status.value = "<span style='color:%s'>Recomputing signals + confidence…</span>" % MUTED
    res = build_signals(STATE["px"], STATE["iv"], current_cfg())
    STATE["res"] = res
    signals_box.children = [render_signals(res)]
    iv_table_box.children = [render_iv_table(res)]

    corr = res["engines"].get("Correlation RV")
    disp = res["engines"].get("Vol dispersion (pairs)")
    parts = []
    for label, df in [("Correlation dislocations", corr), ("Vol-spread dislocations", disp)]:
        if df is not None and not df.empty:
            items = "".join(
                "<div style='padding:5px 0;border-top:1px solid %s;font:400 12px Inter,Arial;color:%s'>"
                "<b style='color:%s'>%s</b> &nbsp;—&nbsp; %s &nbsp;—&nbsp; <b>%s</b> &nbsp;—&nbsp; "
                "<span style='color:%s;font-weight:700'>%.0f%% conf</span></div>"
                % (GRID, TXT, TXT, r["name"], r["trigger"], r["side"],
                   _conf_color(r.get("conf", np.nan)), r.get("conf", 50.0))
                for _, r in df.iterrows())
            parts.append("<div style='font:700 11px Inter,Arial;color:%s;letter-spacing:.06em;"
                         "margin:8px 0 2px'>%s</div>%s" % (MUTED, label.upper(), items))
    disloc_box.children = [widgets.HTML("".join(parts) if parts else
                           "<div style='color:%s'>Nothing above threshold.</div>" % MUTED)]

    redraw_iv_chart()
    redraw_pair_chart()
    total = sum(len(res["engines"][n]) for n, _ in ENGINE_ORDER)
    status.value = "<span style='color:%s'>Ready · %d signals.</span>" % (GREEN, total)


def load(*_):
    status.value = "<span style='color:%s'>Fetching from Bloomberg…</span>" % MUTED
    try:
        px, iv, px_fail, iv_fail = fetch_all(status)
        STATE["px"], STATE["iv"] = px, iv
        iv_drill_dd.options = grouped_options(list(iv.columns))
        pair_opts = grouped_options(list(px["close"].columns))
        pair_a_dd.options = pair_opts
        pair_b_dd.options = pair_opts
        first_iv = [t for _, t in iv_drill_dd.options if t is not None]
        first_px = [t for _, t in pair_opts if t is not None]
        if first_iv:
            iv_drill_dd.value = first_iv[0]
        if len(first_px) > 1:
            pair_a_dd.value, pair_b_dd.value = first_px[0], first_px[1]
        recompute()
        warn = ""
        if iv_fail:
            warn = "<br><span style='color:%s'>No IV: %s</span>" % (
                AMBER, ", ".join(NAME.get(t, t) for t in iv_fail))
        if px_fail:
            warn += "<br><span style='color:%s'>No price: %s</span>" % (
                RED, ", ".join(NAME.get(t, t) for t in px_fail))
        status.value = ("<span style='color:%s'>Loaded %d price assets, %d IV assets.</span>%s"
                        % (GREEN, px["close"].shape[1], iv.shape[1], warn))
    except Exception as e:
        status.value = "<span style='color:%s'>Error: %s</span>" % (RED, e)


def redraw_iv_chart(*_):
    if "res" in STATE and iv_drill_dd.value:
        iv_chart_box.children = [iv_drill_fig(STATE["res"], iv_drill_dd.value)]


def redraw_pair_chart(*_):
    a, b = pair_a_dd.value, pair_b_dd.value
    if "res" in STATE and a and b and a != b:
        pair_chart_box.children = [corr_pair_fig(STATE["res"], a, b)]
        vol_pair_chart_box.children = [vol_corr_pair_fig(STATE["res"], a, b)]
        leadlag_chart_box.children = [leadlag_pair_fig(STATE["res"], a, b)]


load_btn.on_click(load)
iv_drill_dd.observe(redraw_iv_chart, names="value")
pair_a_dd.observe(redraw_pair_chart, names="value")
pair_b_dd.observe(redraw_pair_chart, names="value")
for w in [lookback_dd, rv_est_dd, rv_win_dd, iv_lo, iv_hi, disp_z, corr_z,
          ll_r, ll_gap, min_quality, exclude_stale, min_conf]:
    w.observe(recompute, names="value")


def hdr(t, sub=""):
    s = ("<div style='font:400 11px Inter,Arial;color:%s;margin:-4px 0 8px'>%s</div>" % (MUTED, sub)) if sub else ""
    return widgets.HTML("<div style='font:700 15px Inter,Arial;color:%s;margin:6px 0 8px'>%s</div>%s"
                        % (TXT, t, s))


controls = widgets.HBox(
    [widgets.VBox([lookback_dd, rv_est_dd, rv_win_dd, min_quality, exclude_stale]),
     widgets.VBox([iv_lo, iv_hi, disp_z, corr_z]),
     widgets.VBox([ll_r, ll_gap, min_conf, load_btn])],
    layout=widgets.Layout(border="1px solid %s" % GRID, border_radius="10px",
                          padding="14px", justify_content="space-between"))

tab_signals = widgets.VBox([hdr("Trade signals"), signals_box])
tab_vol = widgets.VBox([hdr("Vol monitor"), iv_table_box, widgets.HTML("<br>"),
                        iv_drill_dd, iv_chart_box])
tab_corr = widgets.VBox([
    hdr("Correlation & pairs"), disloc_box, widgets.HTML("<br>"),
    widgets.HBox([pair_a_dd, pair_b_dd]),
    hdr("Price-return correlation",
        "How tightly the two move together, and whether that link is currently stretched."),
    pair_chart_box,
    hdr("Implied-vol correlation",
        "Do their option markets re-price risk in tandem? (correlation of daily IV changes)"),
    vol_pair_chart_box,
    hdr("Lead-lag — who moves first",
        "Top: prices rebased, with the leader re-drawn shifted forward by its measured lead — "
        "if the lead is real the dashed line overlays the follower. "
        "Bottom: correlation at every lag; the tallest bar is the lead, and a flat curve means "
        "there is no timing edge, only ordinary correlation."),
    leadlag_chart_box])

tabs = widgets.Tab(children=[tab_signals, tab_vol, tab_corr])
for i, t in enumerate(["Signals", "Vol Monitor", "Correlation & Pairs"]):
    tabs.set_title(i, t)

header = widgets.HTML(
    "<div style='font:800 24px Inter,Segoe UI,Arial;color:%s'>Commodities Vol &amp; Relative-Value Dashboard</div>"
    "<div style='font:400 12px Inter,Arial;color:%s;padding:3px 0 8px'>"
    "Implied vol: hist_put_imp_vol (~30d) &nbsp;·&nbsp; correlation/lead-lag within asset class, "
    "plus cross-asset vs SPX / DXY / 10y &nbsp;·&nbsp; confidence = historical hit rate of the same setup</div>"
    % (TXT, MUTED))

ui = widgets.VBox([header, controls, status, tabs])
load()
ui

---
### Notes
- **Load / refresh data** hits Bloomberg (OHLC + implied vol, 3y history). The sliders and dropdowns
  re-run the analytics on the already-loaded data instantly — there is no separate recompute button.
  Changing a control now also re-runs the confidence hit-rates, so a redraw takes a beat longer.
- Percentile **lookback** (1y/2y/3y) is the history each IV / spread percentile is ranked against,
  and also the trailing window the confidence backtest uses to define "comparable setup".
- Correlation and lead-lag pairs: **same asset class**, plus **anything vs SPX / DXY / 10y**.
- **Min confidence %** hides low-conviction ideas across every engine at once. Left at 0 you see
  everything, including sub-50% signals that historically went the other way — worth a look, since
  a persistently sub-45% setup is really a signal to fade.
- "Vol dispersion" needs both legs to pass the quality filter, so illiquid legs are dropped. The
  macro names carry no implied vol at all, so they never appear in the three vol engines.
- Confidence is an in-sample hit rate on 3 years of one asset's own history — a sanity check on
  whether the setup has ever worked, not a backtested P&L. Small **n** means treat it as unproven.
